# Thành viên 3
Các bảng phụ trách: `returns.csv`, `city.csv`, `orders.csv`, `location.csv`

Chạy lần lượt từ **Ô 0** đến ô cuối. Mỗi ô tạo đúng một file CSV, đọc từ dữ liệu gốc `student_data`.

## Ô 0: Chuẩn bị

In [1]:
from pathlib import Path
import zipfile
import pandas as pd

DATA = Path("/content/student_data")   # dữ liệu gốc (bronze)
OUT = Path("/content/silver")          # kết quả (silver)
OUT.mkdir(exist_ok=True)

if not DATA.exists():                  # chưa có dữ liệu -> chọn file student_data.zip
    from google.colab import files
    for name in files.upload():
        zipfile.ZipFile(name).extractall("/content")

def strip(df, cols):
    """Bỏ khoảng trắng thừa ở các cột chữ."""
    for c in cols:
        df[c] = df[c].astype("string").str.strip()
    return df

def save(df, name, pk):
    """Kiểm tra khóa chính (duy nhất, không rỗng) rồi ghi CSV."""
    pk = [pk] if isinstance(pk, str) else pk
    assert df[pk].notna().all().all() and not df.duplicated(pk).any(), f"{name}: khóa chính lỗi"
    df.to_csv(OUT / f"{name}.csv", index=False, encoding="utf-8-sig")
    print(f"{name}.csv: {len(df):,} dòng, khóa chính {pk} hợp lệ")
    return df.head()

Saving student_data.zip to student_data.zip


## Ô 1: `returns.csv`

In [2]:
returns = strip(pd.read_csv(DATA / "returns.csv"), ["return_id", "return_reason"])
returns["return_reason"] = returns["return_reason"].str.lower()
returns["return_date"] = pd.to_datetime(returns["return_date"])
assert (returns["return_quantity"] >= 1).all() and (returns["refund_amount"] >= 0).all()

save(returns.drop_duplicates("return_id"), "returns", "return_id")

returns.csv: 39,939 dòng, khóa chính ['return_id'] hợp lệ


,return_id,order_id,product_id,return_date,return_reason,return_quantity,refund_amount
0,RET-000001,2,609,2012-07-25,late_delivery,6,52458.01
1,RET-000002,32,1862,2012-07-16,wrong_size,2,5141.37
2,RET-000003,35,2359,2012-07-16,wrong_size,1,5315.95
3,RET-000004,47,1449,2012-07-11,wrong_size,4,6493.75
4,RET-000005,47,1450,2012-07-25,wrong_size,1,1740.76


## Ô 2: `city.csv`

In [3]:
# Sửa: bảng mới. Trong geography, city -> region (mỗi thành phố thuộc đúng 1 miền) nên region tách ra bảng city
geo = strip(pd.read_csv(DATA / "geography.csv"), ["city", "region", "district"])
assert (geo.groupby("city")["region"].nunique() == 1).all(), "city -> region không nhất quán"

city = geo[["city", "region"]].drop_duplicates("city").rename(columns={"city": "city_name"})

save(city.sort_values("city_name").reset_index(drop=True), "city", "city_name")

city.csv: 42 dòng, khóa chính ['city_name'] hợp lệ


,city_name,region
0,Bac Giang,East
1,Bac Lieu,West
2,Bac Ninh,East
3,Ben Tre,West
4,Bien Hoa,West


## Ô 3: `orders.csv`

In [4]:
# Sửa: (1) bỏ cột dư. Nhân viên đã có ở employees, zip suy ra từ customer_id, payment_method đã có ở payments,
#          city/region/district đã có ở location và city  (2) chỉ giữ các cột thuộc về chính đơn hàng
cols = ["order_id", "order_date", "customer_id", "sales_employee_id",
        "order_status", "device_type", "order_source", "comment"]
orders = pd.read_csv(DATA / "orders_enriched.csv", usecols=cols)
strip(orders, ["sales_employee_id", "order_status", "device_type", "order_source", "comment"])
orders["sales_employee_id"] = orders["sales_employee_id"].str.upper()
orders["order_date"] = pd.to_datetime(orders["order_date"])

save(orders.drop_duplicates("order_id").sort_values("order_id").reset_index(drop=True)[cols], "orders", "order_id")

orders.csv: 646,945 dòng, khóa chính ['order_id'] hợp lệ


,order_id,order_date,customer_id,sales_employee_id,order_status,device_type,order_source,comment
0,1,2012-07-04,58578,EMP0103,delivered,desktop,paid_search,"Dịch vụ chăm sóc khách hàng chuyên nghiệp, thá..."
1,2,2012-07-04,58621,EMP0180,returned,mobile,paid_search,"Dịch vụ ổn định, nhân viên luôn sẵn sàng hỗ tr..."
2,3,2012-07-04,58811,EMP0093,delivered,desktop,direct,Khách hàng không có thêm phản hồi về chất lượn...
3,4,2012-07-04,59453,EMP0015,delivered,desktop,referral,"Phản hồi thắc mắc đầy đủ, hỗ trợ xuyên suốt qu..."
4,6,2012-07-06,57821,EMP0107,delivered,mobile,email_campaign,Khách hàng đánh giá mức độ hài lòng ở mức khá.


## Ô 4: `location.csv`

In [5]:
# Sửa: lấy từ geography.csv gốc (đủ 39.948 zip), KHÔNG suy ra từ orders (bản cũ chỉ có 29.932 zip
#      làm 4.917 khách hàng bị mồ côi). Cột region đã chuyển sang bảng city
geo = strip(pd.read_csv(DATA / "geography.csv"), ["city", "district"]).rename(columns={"city": "city_name"})
location = geo[["zip", "city_name", "district"]].drop_duplicates("zip")

save(location.sort_values("zip").reset_index(drop=True), "location", "zip")

location.csv: 39,948 dòng, khóa chính ['zip'] hợp lệ


,zip,city_name,district
0,1,Da Lat,District #34
1,2,Rach Gia,District #34
2,3,Long Xuyen,District #34
3,4,Soc Trang,District #34
4,5,My Tho,District #34
